# 02 - Silver Layer

## Procurement Analytics Pipeline

The Silver Layer contains cleaned, standardized, and deduplicated procurement data prepared for analytical processing.

### Objectives

* Handle missing values.
* Remove duplicate records.
* Standardize timestamps.
* Apply data quality rules.
* Prepare vendor contracts for SCD Type 2 processing.
* Store curated datasets in the Silver Layer.


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, trim, current_date, lit

spark = SparkSession.builder \
    .appName('Silver Layer') \
    .getOrCreate()

print('Spark Started')

Spark Started


## Load Bronze Layer Datasets

Load the raw Bronze CSV files generated in the previous notebook.


In [15]:
orders_df = spark.read.option('header', True).csv('output/bronze/orders.csv')

invoices_df = spark.read.option('header', True).csv('output/bronze/invoices.csv')

vendors_df = spark.read.option('header', True).csv('output/bronze/vendors.csv')

contracts_df = spark.read.option('header', True).csv('output/bronze/contracts.csv')

print('Bronze files loaded successfully')

Bronze files loaded successfully


## Clean Purchase Orders

Remove invalid rows and standardize timestamps.


In [16]:
# Trim item names
orders_clean = orders_df.withColumn(
    'item_name',
    trim(col('item_name'))
)

# Replace invalid date strings with NULL
orders_clean = orders_clean.withColumn(
    'po_timestamp_clean',
    when(
        col('po_timestamp') == 'N/A - Unknown Date',
        None
    ).otherwise(col('po_timestamp'))
)

# Safe timestamp conversion
orders_clean = orders_clean.withColumn(
    'po_timestamp',
    to_timestamp(col('po_timestamp_clean'))
).drop('po_timestamp_clean')

# Remove invalid rows
orders_silver = orders_clean.filter(
    col('item_name').isNotNull()
).filter(
    col('po_timestamp').isNotNull()
)

print('Orders after cleaning:', orders_silver.count())
orders_silver.show(5, truncate=False)

Orders after cleaning: 4618
+---------+---------+-------------+------------------+-------------------+--------------------------+
|po_id    |vendor_id|item_name    |quantity_requested|po_timestamp       |ingestion_timestamp       |
+---------+---------+-------------+------------------+-------------------+--------------------------+
|PO0000001|V00637   |E-markets    |487               |2025-04-08 15:51:30|2026-08-12 21:41:41.391515|
|PO0000002|V01075   |Channels     |326               |2025-05-21 20:44:55|2026-08-12 21:41:41.391515|
|PO0000003|V03667   |Models       |24                |2024-05-20 14:26:10|2026-08-12 21:41:41.391515|
|PO0000004|V01296   |Architectures|756               |2024-10-18 02:03:33|2026-08-12 21:41:41.391515|
|PO0000005|V04012   |Bandwidth    |554               |2025-10-11 07:07:53|2026-08-12 21:41:41.391515|
+---------+---------+-------------+------------------+-------------------+--------------------------+
only showing top 5 rows


## Clean Invoices

### Business Rules

* Remove duplicate invoice records.
* Convert invoice timestamps to timestamp type.


In [17]:
invoices_silver = invoices_df.dropDuplicates() \
    .withColumn(
        'invoice_timestamp',
        to_timestamp(col('invoice_timestamp'))
    )

print('Invoices after cleaning:', invoices_silver.count())
invoices_silver.show(5, truncate=False)

Invoices after cleaning: 5000
+-----------+---------+-----------------------+-------------------+--------------------------+
|invoice_id |po_id    |invoiced_price_per_unit|invoice_timestamp  |ingestion_timestamp       |
+-----------+---------+-----------------------+-------------------+--------------------------+
|INV00000238|PO0000108|4284.73                |2026-02-06 22:25:46|2026-08-12 21:45:25.315550|
|INV00000267|PO0002012|2872.58                |2025-11-23 08:49:32|2026-08-12 21:45:25.315550|
|INV00000646|PO0004082|86.63                  |2025-05-15 15:37:07|2026-08-12 21:45:25.315550|
|INV00001379|PO0002603|1450.19                |2024-08-23 07:56:07|2026-08-12 21:45:25.315550|
|INV00001669|PO0004233|623.86                 |2025-03-03 12:47:30|2026-08-12 21:45:25.315550|
+-----------+---------+-----------------------+-------------------+--------------------------+
only showing top 5 rows


## Clean Vendors

### Business Rules

* Remove duplicate vendors based on `vendor_id`.
* Trim vendor names.


In [18]:
vendors_silver = vendors_df.dropDuplicates(['vendor_id']) \
    .withColumn(
        'vendor_name',
        trim(col('vendor_name'))
    )

print('Vendors after cleaning:', vendors_silver.count())
vendors_silver.show(5, truncate=False)

Vendors after cleaning: 5000
+---------+-------------------------------+------+-----------+--------------------------+
|vendor_id|vendor_name                    |region|risk_rating|ingestion_timestamp       |
+---------+-------------------------------+------+-----------+--------------------------+
|V00001   |Rodriguez, Figueroa and Sanchez|AMER  |Low        |2026-08-12 21:45:32.070184|
|V00002   |Doyle Ltd                      |EMEA  |High       |2026-08-12 21:45:32.070184|
|V00003   |Mcclain, Miller and Henderson  |AMER  |High       |2026-08-12 21:45:32.070184|
|V00004   |Davis and Sons                 |EMEA  |Low        |2026-08-12 21:45:32.070184|
|V00005   |Guzman, Hoffman and Baldwin    |EMEA  |High       |2026-08-12 21:45:32.070184|
+---------+-------------------------------+------+-----------+--------------------------+
only showing top 5 rows


## Clean Vendor Contracts

### Business Rules

* Remove duplicate contracts based on `contract_id`.
* Trim item names.
* Convert contract validity dates to timestamp type.


In [28]:
from pyspark.sql.functions import col, trim, when, coalesce, expr

# Remove duplicates and trim text
contracts_clean = contracts_df.dropDuplicates(['contract_id']) \
    .withColumn('item_name', trim(col('item_name'))) \
    .withColumn('valid_from_raw', trim(col('valid_from')))

# Replace empty strings with NULL
contracts_clean = contracts_clean.withColumn(
    'valid_from_raw',
    when(
        (col('valid_from_raw') == '') |
        (col('valid_from_raw').isNull()),
        None
    ).otherwise(col('valid_from_raw'))
)

# Safe parsing for mixed formats
contracts_silver = contracts_clean.withColumn(
    'valid_from',
    coalesce(
        expr("try_to_timestamp(valid_from_raw, 'MM/dd/yyyy')"),
        expr("try_to_timestamp(valid_from_raw, 'yyyy-MM-dd')")
    )
).drop('valid_from_raw')

print('Contracts after cleaning:', contracts_silver.count())

# THIS WILL NOW WORK
contracts_silver.show(5, truncate=False)

Contracts after cleaning: 5000
+-----------+---------+-------------+----------------+-------------------+--------------------------+
|contract_id|vendor_id|item_name    |negotiated_price|valid_from         |ingestion_timestamp       |
+-----------+---------+-------------+----------------+-------------------+--------------------------+
|C000001    |V04273   |Solutions    |1745.93         |2024-04-25 00:00:00|2026-08-12 21:45:41.677177|
|C000002    |V02630   |Relationships|3538.33         |2023-07-05 00:00:00|2026-08-12 21:45:41.677177|
|C000003    |V04966   |Communities  |2303.8          |2025-06-15 00:00:00|2026-08-12 21:45:41.677177|
|C000004    |V03938   |Metrics      |17.46           |2025-11-15 00:00:00|2026-08-12 21:45:41.677177|
|C000005    |V00259   |E-commerce   |4058.43         |2025-08-09 00:00:00|2026-08-12 21:45:41.677177|
+-----------+---------+-------------+----------------+-------------------+--------------------------+
only showing top 5 rows


## Prepare SCD Type 2 Columns

Additional columns are added to support historical contract tracking in the next notebook.


In [29]:
contracts_scd = contracts_silver \
    .withColumn('effective_start_date', col('valid_from')) \
    .withColumn('effective_end_date', lit(None).cast('timestamp')) \
    .withColumn('is_current', lit(True))

contracts_scd.show(5, truncate=False)

+-----------+---------+-------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+
|contract_id|vendor_id|item_name    |negotiated_price|valid_from         |ingestion_timestamp       |effective_start_date|effective_end_date|is_current|
+-----------+---------+-------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+
|C000001    |V04273   |Solutions    |1745.93         |2024-04-25 00:00:00|2026-08-12 21:45:41.677177|2024-04-25 00:00:00 |NULL              |true      |
|C000002    |V02630   |Relationships|3538.33         |2023-07-05 00:00:00|2026-08-12 21:45:41.677177|2023-07-05 00:00:00 |NULL              |true      |
|C000003    |V04966   |Communities  |2303.8          |2025-06-15 00:00:00|2026-08-12 21:45:41.677177|2025-06-15 00:00:00 |NULL              |true      |
|C000004    |V03938   |Metrics      |17.46           |2025-11-15 00:00:00|2026-08-

## Data Quality Validation

Validate that duplicate records have been removed successfully.


In [21]:
print(
    'Vendor duplicates remaining:',
    vendors_silver.count() -
    vendors_silver.dropDuplicates(['vendor_id']).count()
)

print(
    'Contract duplicates remaining:',
    contracts_scd.count() -
    contracts_scd.dropDuplicates(['contract_id']).count()
)

Vendor duplicates remaining: 0
Contract duplicates remaining: 0


## Save Silver Layer

Save all curated Silver datasets as CSV files for downstream processing and GitHub submission.


In [30]:
os.makedirs('output/silver', exist_ok=True)

orders_silver.toPandas().to_csv(
    'output/silver/orders_silver.csv',
    index=False
)

invoices_silver.toPandas().to_csv(
    'output/silver/invoices_silver.csv',
    index=False
)

vendors_silver.toPandas().to_csv(
    'output/silver/vendors_silver.csv',
    index=False
)

contracts_scd.toPandas().to_csv(
    'output/silver/contracts_silver.csv',
    index=False
)

print('All Silver files saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow

All Silver files saved successfully


d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Verify Saved Files

Confirm that all Silver layer files were created successfully.


In [31]:
print(os.listdir('output/silver'))

['contracts_silver.csv', 'invoices_silver.csv', 'orders_silver.csv', 'vendors_silver.csv']


## Silver Layer Summary

| Dataset   | Bronze Rows | Silver Rows | Action                 |
| --------- | ----------- | ----------- | ---------------------- |
| Orders    | 5000        | 4709        | Invalid rows removed   |
| Invoices  | 5000        | 5000        | Standardized           |
| Vendors   | 5100        | 5000        | 100 duplicates removed |
| Contracts | 5050        | 5000        | 50 duplicates removed  |


# Conclusion

The Silver Layer was successfully created by cleaning and standardizing procurement datasets. Invalid purchase order records were removed, duplicate vendor and contract records were eliminated, timestamps were standardized, and SCD tracking columns were added to vendor contracts.

The curated Silver datasets are now ready for SCD Type 2 processing and Gold-layer business analytics.
